# TTM R3 compatibility probe
Run in a fresh Colab Pro+ runtime. This is a shape/API smoke test on synthetic data and one ETTh1 validation window, not a benchmark. It never reads the test split.

In [ ]:
# After cloning this repository and checking out the intended project commit:
%pip install -q -r requirements/ttm.txt
!git rev-parse HEAD
!python -m pip freeze

In [ ]:
import hashlib, json, os, platform, tempfile, time
from pathlib import Path
import pandas as pd
import torch
from tsfm_public.models.tinytimemixer import TinyTimeMixerForDecomposedPrediction
os.environ.setdefault('HF_HOME', '.cache/ttm')
REPO = 'ibm-granite/granite-timeseries-ttm-r3'
REVISION = '7b070728ff280ee2bb682bd8dfbe71da21ca2c7c'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print({'python': platform.python_version(), 'torch': torch.__version__, 'device': str(DEVICE)})

In [ ]:
def parameter_digest(model):
    h = hashlib.sha256()
    for name, value in sorted(model.state_dict().items()):
        h.update(name.encode()); h.update(value.detach().cpu().numpy().tobytes())
    return h.hexdigest()
model = TinyTimeMixerForDecomposedPrediction.from_pretrained(REPO, revision=REVISION, prediction_filter_length=96, cache_dir='.cache/ttm').to(DEVICE)
model.eval()
x = torch.randn(1, 512, 7, device=DEVICE)
before = parameter_digest(model)
started = time.perf_counter()
with torch.no_grad(): y1 = model(past_values=x).prediction_outputs
with torch.no_grad(): y2 = model(past_values=x).prediction_outputs
elapsed = time.perf_counter() - started
assert y1.shape == (1, 96, 7) and torch.isfinite(y1).all()
assert parameter_digest(model) == before
synthetic = {'shape': list(y1.shape), 'dtype': str(y1.dtype), 'bitwise_repeat': torch.equal(y1, y2), 'seconds_two_forwards': elapsed, 'parameters': sum(p.numel() for p in model.parameters())}
print(synthetic)

In [ ]:
path = Path('data/official_raw/ett/ETTh1.csv')
assert path.exists(), 'Place the official ignored ETTh1 file at the documented path.'
frame = pd.read_csv(path); values = frame.drop(columns=['date']).to_numpy(dtype='float32')
train_end, val_end = int(.6*len(values)), int(.8*len(values))
origin = train_end + 512
assert origin + 96 <= val_end
past = torch.from_numpy(values[origin-512:origin]).unsqueeze(0).to(DEVICE)
future = torch.from_numpy(values[origin:origin+96]).unsqueeze(0).to(DEVICE)
model.eval(); pre = parameter_digest(model)
with torch.no_grad(): validation_prediction = model(past_values=past).prediction_outputs
assert validation_prediction.shape == future.shape and torch.isfinite(validation_prediction).all()
assert parameter_digest(model) == pre
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6)
model.train(); optimizer.zero_grad(set_to_none=True)
output = model(past_values=past, future_values=future)
assert torch.isfinite(output.loss); output.loss.backward()
assert any(p.grad is not None for p in model.parameters() if p.requires_grad)
optimizer.step(); assert parameter_digest(model) != pre
trained_hash=parameter_digest(model)
with tempfile.TemporaryDirectory() as directory:
    state_path=Path(directory)/'training_state.pt'; torch.save({'model':model.state_dict(),'optimizer':optimizer.state_dict(),'step':1},state_path)
    with torch.no_grad(): next(model.parameters()).add_(1.0)
    state=torch.load(state_path,map_location=DEVICE,weights_only=False); model.load_state_dict(state['model']); optimizer.load_state_dict(state['optimizer'])
    assert state['step']==1 and parameter_digest(model)==trained_hash
model.eval();
with torch.no_grad(): restored_validation = model(past_values=past, future_values=future)
assert torch.isfinite(restored_validation.loss)
runtime = {'model': REPO, 'revision': REVISION, 'device': str(DEVICE), 'synthetic': synthetic, 'validation_window_origin': origin, 'zero_shot_parameter_hash_unchanged': True, 'finetune_steps': 1, 'finetune_loss_finite': True, 'training_state_round_trip': True, 'test_split_used': False}
Path('results/manifests/models').mkdir(parents=True, exist_ok=True)
Path('results/manifests/models/ttm_runtime_probe.json').write_text(json.dumps(runtime, indent=2), encoding='utf-8')
runtime